# Multimodal Skin Lesion Classification on HAM10000: Progressive Feature Engineering, Hybrid Deep Learning, and Clinical Decision Optimization

**Target Journal:** *Biomedical Signal Processing and Control* (Elsevier)  
**Execution Environment:** Kaggle GPU (1x or 2x NVIDIA Tesla T4 / P100) or Local GPU/CPU

---

### Complete Experimental Protocol (Addressing All 7 Advisor Priorities):
1. **Strict Data Leakage Prevention (Priority 1.3):**
   - Full ingestion of the HAM10000 dataset (`marmal88/skin_cancer`) preserving `lesion_id`.
   - Partitioning into **70% Train / 15% Validation / 15% Test** via `StratifiedGroupKFold` grouped strictly by `lesion_id` with zero lesion overlap assertions.
   - `VarianceThreshold`, `StandardScaler`, class weights, and SMOTEENN are fitted **exclusively on the training split**.
   - Model selection (`EarlyStopping`, `ReduceLROnPlateau`) and threshold optimization ($\theta_k$) use **only the validation split**. The test split is evaluated strictly once (*hold-out*).
2. **Progressive Handcrafted Feature Engineering & Ablation Study (Priority 1 & 5):**
   - **DullRazor** hair removal (Black-Hat morphology + Telea inpainting) and **Center Crop 1:1** ($224 \times 224$).
   - **ABCD Clinical Vector (30 descriptors):** Color (RGB & HSV means: 6), Texture (GLCM Haralick: 4 + uniform LBP: 13 = 17), and Asymmetry/Shape (Otsu segmentation + 7 log-transformed Hu Moments).
   - Progressive **Random Forest** evaluation (*Color only* $\rightarrow$ *Color + Texture* $\rightarrow$ *Full ABCD*) + **Ablation Study** (with/without DullRazor and ABCD shape descriptors).
3. **Comparative Deep Learning Baselines & Multi-Seed Evaluation (Priority 1.1 & Priority 2):**
   - Standardized comparison across **EfficientNetV2B0 (Visual-Only)**, **EfficientNetV2B0 (Hybrid Image + ABCD)**, **ResNet50**, and **ConvNeXt-Tiny** under identical data splits, preprocessing, Focal Loss ($\gamma=2.0$), balanced sampling, and training budget.
   - Automated multi-seed training (`SEEDS = [42, 100, 2024]`) reporting **Mean $\pm$ SD**.
   - Computational cost profiling: total/trainable parameters, training time per epoch, inference latency (ms/image), and peak GPU VRAM.
4. **Probabilistic Calibration, Threshold Optimization & Statistical Auditing (Priority 1.2, 1.4, 1.5, 1.6 & 2.5):**
   - Explicit distinction between probabilistic calibration (**Multiclass Brier Score**, **Expected Calibration Error - ECE**, and **Temperature Scaling**) and post-hoc clinical decision threshold tuning ($\hat{y} = \arg\max_k (p_k / \theta_k)$).
   - **Grouped Bootstrap (1,000 iterations by `lesion_id`)** for 95% Confidence Intervals across all metrics (Multiclass Accuracy, OvR Accuracy, Balanced Accuracy / Macro Recall, Macro Precision, Macro Specificity, Macro F1, MCC, ROC-AUC, PR-AUC).
   - Paired statistical hypothesis testing: **McNemar's Test** and **Paired Cluster Bootstrap $p$-values**.
5. **Generalization, Robustness & Minority Class Analysis (Priority 3):**
   - Robustness evaluation under realistic dermatoscopic perturbations (brightness, contrast, Gaussian noise, optical blur, JPEG compression).
   - Dedicated minority class audit (`df`, `vasc`, `akiec`, `bcc`) and intra-dataset acquisition shift analysis.
6. **Clinical Error Analysis & Explainability (Priority 4 & 6):**
   - Absolute and row-normalized Confusion Matrices (300 DPI PDF/PNG).
   - Detailed audit of **Melanoma False Negatives (FN)** vs. True Positives (TP) and False Positives (FP).
   - Automated **Grad-CAM** visual explanations across representative TP, FP, and FN cases.
   - Automatic export of LaTeX tables, figures, weights, and reproducibility metadata to `/kaggle/working/bspc_submission_artifacts/`.

In [ ]:
import os
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

!pip install datasets imbalanced-learn albumentations scikit-image -q

import gc
import sys
import time
import json
import platform
import warnings
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset, concatenate_datasets
from skimage.color import rgb2hsv, rgb2lab
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from skimage.filters import threshold_otsu
from skimage.measure import moments, moments_hu, label, regionprops

from sklearn.feature_selection import VarianceThreshold
from sklearn.manifold import TSNE
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    auc,
)
from imblearn.combine import SMOTEENN

import tensorflow as tf
from tensorflow.keras import callbacks, layers, models, optimizers, mixed_precision

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.family"] = "DejaVu Sans"

# Output directory for paper figures, LaTeX tables, and reproducibility logs
ARTIFACT_DIR = Path("/kaggle/working/bspc_submission_artifacts") if Path("/kaggle/working").exists() else Path("./bspc_submission_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Experimental Configuration
MASTER_SEED = 42
MULTI_SEEDS = [42, 100, 2024]  # Priority 2.3: Multi-seed evaluation
IMG_SIZE = (224, 224)
EFFECTIVE_BATCH_SIZE = 32
WARMUP_EPOCHS = 6
FINETUNE_EPOCHS = 16
N_BOOTSTRAP = 1000

def set_global_seed(seed=MASTER_SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

set_global_seed(MASTER_SEED)

print("--- Configuring Hardware & Memory Policy ---")
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception:
            pass

USE_SINGLE_GPU = os.environ.get("USE_SINGLE_GPU", "false").lower() == "true"
if len(gpus) > 1 and not USE_SINGLE_GPU:
    strategy = tf.distribute.MirroredStrategy()
    PER_REPLICA_BATCH = 16
    BATCH_SIZE = PER_REPLICA_BATCH * strategy.num_replicas_in_sync
    ACCUM_STEPS = 1
    print(f"Hardware: {len(gpus)} GPUs Active (MirroredStrategy) | Global Batch = {BATCH_SIZE}")
elif len(gpus) >= 1:
    strategy = tf.distribute.OneDeviceStrategy(device="/gpu:0")
    PER_REPLICA_BATCH = 16
    BATCH_SIZE = PER_REPLICA_BATCH
    ACCUM_STEPS = max(1, EFFECTIVE_BATCH_SIZE // BATCH_SIZE)
    print(f"Hardware: 1 GPU Active | Micro-Batch = {BATCH_SIZE} | Accum Steps = {ACCUM_STEPS} (Effective = {BATCH_SIZE * ACCUM_STEPS})")
else:
    strategy = tf.distribute.get_strategy()
    BATCH_SIZE = 16
    ACCUM_STEPS = 2
    print("Hardware: CPU Mode")

try:
    mixed_precision.set_global_policy("mixed_float16")
    print("Mixed Precision (float16) enabled.")
except Exception as e:
    print(f"Mixed Precision not enabled: {e}")

try:
    _sample_from_datasets = tf.data.Dataset.sample_from_datasets
except AttributeError:
    _sample_from_datasets = tf.data.experimental.sample_from_datasets

def get_gpu_memory_mb():
    if not gpus:
        return {"current_mb": 0.0, "peak_mb": 0.0}
    try:
        info = tf.config.experimental.get_memory_info("GPU:0")
        return {
            "current_mb": round(info["current"] / (1024 ** 2), 2),
            "peak_mb": round(info["peak"] / (1024 ** 2), 2),
        }
    except Exception:
        return {"current_mb": 0.0, "peak_mb": 0.0}

env_metadata = {
    "python_version": sys.version,
    "platform": platform.platform(),
    "tensorflow_version": tf.__version__,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "opencv_version": cv2.__version__,
    "num_gpus": len(gpus),
    "gpu_names": [g.name for g in gpus],
    "img_size": IMG_SIZE,
    "effective_batch_size": BATCH_SIZE * ACCUM_STEPS,
    "warmup_epochs": WARMUP_EPOCHS,
    "finetune_epochs": FINETUNE_EPOCHS,
    "seeds": MULTI_SEEDS,
}
with open(ARTIFACT_DIR / "environment_metadata.json", "w") as f:
    json.dump(env_metadata, f, indent=2)
print("Environment metadata logged to:", ARTIFACT_DIR / "environment_metadata.json")

## 1. Data Ingestion, Hair Artifact Removal (DullRazor), and Lesion-Grouped Partitioning (Priority 1.3)
We ingest all available splits of the HAM10000 dataset (`marmal88/skin_cancer`), retaining both the raw center-cropped images ($224 \times 224$) and the **DullRazor**-filtered images so that we can conduct a rigorous **Ablation Study** on hair removal.
Crucially, we Partition the dataset into **70% Train / 15% Validation / 15% Test** grouped strictly by `lesion_id` using `StratifiedGroupKFold`, and verify zero overlap across splits.

In [ ]:
def dull_razor(img_uint8):
    """Removes hair artifacts using a 9x9 morphological Black-Hat filter and Telea inpainting."""
    gray = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 9))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    _, mask = cv2.threshold(blackhat, 10, 255, cv2.THRESH_BINARY)
    dst = cv2.inpaint(img_uint8, mask, 1, cv2.INPAINT_TELEA)
    return dst

print("--- Loading HAM10000 Dataset + lesion_id ---")
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    print("Hugging Face token detected.")
except Exception:
    hf_token = os.environ.get("HF_TOKEN", None)

# Load all available splits in marmal88/skin_cancer to ensure complete HAM10000 coverage
ds_dict = load_dataset("marmal88/skin_cancer", token=hf_token)
if isinstance(ds_dict, dict):
    all_splits = [ds_dict[k] for k in ds_dict.keys()]
    full_ds = concatenate_datasets(all_splits) if len(all_splits) > 1 else all_splits[0]
else:
    full_ds = ds_dict

N_TOTAL = len(full_ds)
print(f"Total samples found in dataset: {N_TOTAL}")

# Pre-allocate NumPy arrays to keep RAM usage strictly bounded on Kaggle
X_raw_clean = np.empty((N_TOTAL, IMG_SIZE[0], IMG_SIZE[1], 3), dtype=np.uint8)  # DullRazor applied
X_raw_no_dr = np.empty((N_TOTAL, IMG_SIZE[0], IMG_SIZE[1], 3), dtype=np.uint8)  # Without DullRazor (for ablation)
y_labels = []
lesion_ids = []
dx_types = []

for i, example in enumerate(full_ds):
    raw_img = example["image"].convert("RGB")
    w, h = raw_img.size
    min_dim = min(w, h)
    left = (w - min_dim) // 2
    top = (h - min_dim) // 2
    img_cropped = raw_img.crop((left, top, left + min_dim, top + min_dim))
    img_resized = img_cropped.resize(IMG_SIZE)
    img_np = np.array(img_resized, dtype=np.uint8)
    
    X_raw_no_dr[i] = img_np
    X_raw_clean[i] = dull_razor(img_np)
    y_labels.append(example["dx"])
    lesion_ids.append(str(example.get("lesion_id", f"lesion_{i}")))
    dx_types.append(str(example.get("dx_type", "unknown")))
    if (i + 1) % 2000 == 0 or (i + 1) == N_TOTAL:
        print(f"Processed: {i + 1}/{N_TOTAL} images")

del full_ds, ds_dict
gc.collect()

y_labels = np.array(y_labels)
lesion_ids = np.array(lesion_ids)
dx_types = np.array(dx_types)

# Standard short diagnostic codes matching HAM10000 literature
CLASS_NAME_TO_CODE = {
    "actinic_keratoses": "AKIEC",
    "basal_cell_carcinoma": "BCC",
    "benign_keratosis-like_lesions": "BKL",
    "dermatofibroma": "DF",
    "melanocytic_Nevi": "NV",
    "melanoma": "MEL",
    "vascular_lesions": "VASC",
}
y_codes = np.array([CLASS_NAME_TO_CODE.get(lbl, lbl) for lbl in y_labels])

le = LabelEncoder()
y_enc = le.fit_transform(y_codes)
CLASS_CODES = list(le.classes_)
N_CLASSES = len(CLASS_CODES)
MEL_IDX = CLASS_CODES.index("MEL") if "MEL" in CLASS_CODES else 0
NV_IDX = CLASS_CODES.index("NV") if "NV" in CLASS_CODES else 0

print(f"Classes ({N_CLASSES}): {CLASS_CODES}")
print(f"Total Images: {len(y_enc)} | Unique Lesions (lesion_id): {len(np.unique(lesion_ids))}")
for code in CLASS_CODES:
    cnt = int(np.sum(y_codes == code))
    n_les = len(np.unique(lesion_ids[y_codes == code]))
    print(f"  {code:5s}: {cnt:5d} images ({cnt/len(y_enc)*100:5.2f}%) across {n_les:4d} unique lesions")

# --- STRICT 70% / 15% / 15% LESION-GROUPED STRATIFIED SPLIT (Priority 1.3) ---
def create_lesion_grouped_splits(y, groups, seed=MASTER_SEED):
    """
    Uses a 20-fold StratifiedGroupKFold (each fold ~ 5% of dataset) to build
    exact 70% Train (14 folds) / 15% Validation (3 folds) / 15% Test (3 folds)
    partitions with zero lesion_id overlap.
    """
    sgkf = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=seed)
    folds = [val_idx for _, val_idx in sgkf.split(np.zeros(len(y)), y, groups=groups)]
    test_idx = np.concatenate(folds[0:3])      # 3/20 = 15%
    val_idx = np.concatenate(folds[3:6])       # 3/20 = 15%
    train_idx = np.concatenate(folds[6:20])    # 14/20 = 70%
    
    # Audit zero data leakage by lesion_id
    train_lesions = set(groups[train_idx])
    val_lesions = set(groups[val_idx])
    test_lesions = set(groups[test_idx])
    assert len(train_lesions & val_lesions) == 0, "LEAKAGE DETECTED: Train and Val share lesion_id!"
    assert len(train_lesions & test_lesions) == 0, "LEAKAGE DETECTED: Train and Test share lesion_id!"
    assert len(val_lesions & test_lesions) == 0, "LEAKAGE DETECTED: Val and Test share lesion_id!"
    return train_idx, val_idx, test_idx

train_idx, val_idx, test_idx = create_lesion_grouped_splits(y_enc, lesion_ids, seed=MASTER_SEED)
print("\n--- Data Leakage Audit Passed (0 overlapping lesion_ids across splits) ---")
print(f"Train Split : {len(train_idx):5d} images ({len(train_idx)/len(y_enc)*100:.1f}%) | {len(np.unique(lesion_ids[train_idx])):4d} lesions")
print(f"Val Split   : {len(val_idx):5d} images ({len(val_idx)/len(y_enc)*100:.1f}%) | {len(np.unique(lesion_ids[val_idx])):4d} lesions")
print(f"Test Split  : {len(test_idx):5d} images ({len(test_idx)/len(y_enc)*100:.1f}%) | {len(np.unique(lesion_ids[test_idx])):4d} lesions")

## 2. Handcrafted Feature Extraction (ABCD Rule) & Leak-Free Preprocessing
We extract the 30-dimensional clinical feature vector from both the DullRazor-cleaned images (`X_feat_dr`) and the raw uncleaned images (`X_feat_nodr` for the ablation study):
- **Color (C — 6 features):** Mean RGB (3) + Mean HSV (3).
- **Border & Texture (B — 17 features):** 4 Haralick GLCM descriptors (*contrast*, *homogeneity*, *energy*, *correlation*) + 13-bin normalized uniform Local Binary Pattern ($LBP_{8,1}^{u2}$) histogram.
- **Asymmetry & Morphology (A — 7 features):** Otsu adaptive binary lesion mask + 7 log-transformed Hu invariant moments ($\operatorname{sgn}(h_i)\log_{10}(|h_i| + 10^{-12})$).

`VarianceThreshold` and `StandardScaler` are fitted **strictly on `train_idx`**.

In [ ]:
FEATURE_NAMES = (
    ["RGB_R_mean", "RGB_G_mean", "RGB_B_mean", "HSV_H_mean", "HSV_S_mean", "HSV_V_mean"]
    + ["GLCM_contrast", "GLCM_homogeneity", "GLCM_energy", "GLCM_correlation"]
    + [f"LBP_bin_{b}" for b in range(13)]
    + [f"Hu_Moment_{m+1}" for m in range(7)]
)

def extract_all_features(img_uint8):
    img_f = img_uint8.astype("float32") / 255.0
    img_gray = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)
    
    # 1. COLOR (RGB + HSV means: 6 features)
    color_feat = np.concatenate([img_f.mean(axis=(0, 1)), rgb2hsv(img_f).mean(axis=(0, 1))])
    
    # 2. TEXTURE (GLCM 4 features + LBP 13 bins = 17 features)
    glcm = graycomatrix(img_gray, [1], [0], levels=256, symmetric=True, normed=True)
    texture_glcm = [
        graycoprops(glcm, "contrast")[0, 0],
        graycoprops(glcm, "homogeneity")[0, 0],
        graycoprops(glcm, "energy")[0, 0],
        graycoprops(glcm, "correlation")[0, 0],
    ]
    lbp = local_binary_pattern(img_gray, 8, 1, method="uniform")
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, 14), range=(0, 13))
    texture_lbp = hist.astype("float32") / (hist.sum() + 1e-7)
    
    # 3. SHAPE & ASYMMETRY (Otsu Mask + 7 Log Hu Moments)
    try:
        thresh = threshold_otsu(img_gray)
        binary = (img_gray > thresh).astype(np.uint8)
        if binary[0, 0] == 1:
            binary = 1 - binary
        hu = moments_hu(moments(binary))
        hu = -np.sign(hu) * np.log10(np.abs(hu) + 1e-12)
    except Exception:
        hu = np.zeros(7, dtype=np.float32)
        
    return np.concatenate([color_feat, texture_glcm, texture_lbp, hu]).astype(np.float32)

print("Extracting 30 ABCD descriptors (with DullRazor)...")
t0 = time.time()
X_feat_dr = np.nan_to_num(np.array([extract_all_features(img) for img in X_raw_clean], dtype=np.float32))
feat_time_per_img_ms = ((time.time() - t0) / len(X_raw_clean)) * 1000.0
print(f"Extracted {X_feat_dr.shape} in {feat_time_per_img_ms:.2f} ms/image")

print("Extracting 30 ABCD descriptors (without DullRazor, for ablation study)...")
X_feat_nodr = np.nan_to_num(np.array([extract_all_features(img) for img in X_raw_no_dr], dtype=np.float32))

# Delete uncleaned raw images after extracting their features to free 1.4 GB RAM
del X_raw_no_dr
gc.collect()

# --- LEAK-FREE TABULAR SCALING (Fit strictly on Train Split!) ---
vt_selector = VarianceThreshold()
X_tr_vt = vt_selector.fit_transform(X_feat_dr[train_idx])
X_val_vt = vt_selector.transform(X_feat_dr[val_idx])
X_ts_vt = vt_selector.transform(X_feat_dr[test_idx])

scaler = StandardScaler()
f_train_scaled = scaler.fit_transform(X_tr_vt).astype(np.float32)
f_val_scaled = scaler.transform(X_val_vt).astype(np.float32)
f_test_scaled = scaler.transform(X_ts_vt).astype(np.float32)

SELECTED_FEATURE_NAMES = [name for name, keep in zip(FEATURE_NAMES, vt_selector.get_support()) if keep]
N_TABULAR_FEATURES = f_train_scaled.shape[1]
print(f"Active tabular features after leak-free VarianceThreshold: {N_TABULAR_FEATURES}")

## 3. Unified Evaluation Engine: Metrics, Grouped Bootstrap (95% CI), Probabilistic Calibration (Brier & ECE), and Threshold Optimization (Priority 1)

Here we define the mathematical and statistical audit functions required by Priority 1:
1. **Multiclass vs. One-vs-Rest (OvR) Metrics:** Explicitly separates global multiclass accuracy ($\frac{1}{N}\sum_k TP_k$) from mean binary OvR accuracy ($\frac{1}{K}\sum_k \frac{TP_k + TN_k}{N}$) and macro recall ($\text{BA}_{\text{multi}}$) from mean OvR balanced accuracy.
2. **Threshold Optimization ($\theta_k$):**
   $$\hat{y} = \arg\max_{k \in \{1, \dots, K\}} \left( \frac{p_k}{\theta_k} \right)$$
   calibrated **exclusively on the validation set** via coordinate ascent to optimize the clinical F1/Recall trade-off while enforcing high melanoma sensitivity.
3. **Grouped Bootstrap (`lesion_id` cluster resampling):** 1,000 iterations preserving all images of each sampled lesion together.
4. **Probabilistic Calibration:** Multiclass Brier Score, Expected Calibration Error (ECE, 10 bins), and post-hoc Temperature Scaling comparison.
5. **Paired Statistical Tests (Priority 2.5):** McNemar's test and Paired Cluster Bootstrap for model comparisons.

In [ ]:
# --- 3.1 Exploratory t-SNE on Standardized Training Features ---
print("Generating t-SNE projection of handcrafted feature space...")
tsne = TSNE(n_components=2, perplexity=30, max_iter=1000, random_state=MASTER_SEED)
X_tsne = tsne.fit_transform(f_train_scaled)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_enc[train_idx], cmap="tab10", alpha=0.55, s=12)
handles, _ = scatter.legend_elements()
ax.legend(handles, CLASS_CODES, title="Diagnosis", loc="best", frameon=True)
ax.set_title("t-SNE Projection of Handcrafted ABCD Feature Space (Training Set)")
ax.set_xlabel("t-SNE Dimension 1")
ax.set_ylabel("t-SNE Dimension 2")
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "tsne_feature_space.pdf")
plt.savefig(ARTIFACT_DIR / "tsne_feature_space.png")
plt.show()

# --- 3.2 Complete Multiclass & OvR Metric Computation (Priority 1.4) ---
def compute_complete_metrics(y_true, y_pred, y_prob=None):
    """
    Computes exact Multiclass and One-vs-Rest (OvR) metrics across all K classes.
    Returns both a global summary dict and a per-class DataFrame.
    """
    cm = confusion_matrix(y_true, y_pred, labels= np.arange(N_CLASSES))
    N = cm.sum()
    
    per_class = []
    tprs, tnrs, ppvs, npvs, f1s, accs_ovr, bal_accs_ovr, mccs_ovr = [], [], [], [], [], [], [], []
    roc_aucs, pr_aucs = [], []
    
    y_true_bin = label_binarize(y_true, classes=np.arange(N_CLASSES))
    
    for k in range(N_CLASSES):
        tp = int(cm[k, k])
        fn = int(cm[k, :].sum() - tp)
        fp = int(cm[:, k].sum() - tp)
        tn = int(N - (tp + fn + fp))
        
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        tnr = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        npv = tn / (tn + fn) if (tn + fn) > 0 else 0.0
        f1 = (2.0 * tp) / (2.0 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0.0
        acc_ovr = (tp + tn) / N if N > 0 else 0.0
        bal_ovr = 0.5 * (tpr + tnr)
        
        den = np.sqrt(float(tp + fp) * float(tp + fn) * float(tn + fp) * float(tn + fn))
        mcc_k = ((tp * tn) - (fp * fn)) / den if den > 0 else 0.0
        
        if y_prob is not None and y_true_bin[:, k].sum() > 0:
            r_auc = float(roc_auc_score(y_true_bin[:, k], y_prob[:, k]))
            p_auc = float(average_precision_score(y_true_bin[:, k], y_prob[:, k]))
        else:
            r_auc, p_auc = np.nan, np.nan
            
        tprs.append(tpr); tnrs.append(tnr); ppvs.append(ppv); npvs.append(npv)
        f1s.append(f1); accs_ovr.append(acc_ovr); bal_accs_ovr.append(bal_ovr); mccs_ovr.append(mcc_k)
        roc_aucs.append(r_auc); pr_aucs.append(p_auc)
        
        per_class.append({
            "Code": CLASS_CODES[k],
            "Support": int(tp + fn),
            "TP": tp, "FN": fn, "FP": fp, "TN": tn,
            "Precision": ppv,
            "Recall_TPR": tpr,
            "F1_Score": f1,
            "Specificity_TNR": tnr,
            "NPV": npv,
            "Acc_OvR": acc_ovr,
            "BalAcc_OvR": bal_ovr,
            "MCC_OvR": mcc_k,
            "ROC_AUC": r_auc,
            "PR_AUC": p_auc,
        })
        
    df_per_class = pd.DataFrame(per_class)
    
    summary = {
        "Acc_Multi": float(accuracy_score(y_true, y_pred)),
        "Acc_Mean_OvR": float(np.mean(accs_ovr)),
        "BA_Multi_Recall_Macro": float(np.mean(tprs)),
        "BA_Mean_OvR": float(np.mean(bal_accs_ovr)),
        "Precision_Macro": float(np.mean(ppvs)),
        "Specificity_Macro": float(np.mean(tnrs)),
        "F1_Macro": float(np.mean(f1s)),
        "MCC_Macro_OvR": float(np.mean(mccs_ovr)),
        "MCC_Multi": float(matthews_corrcoef(y_true, y_pred)),
        "ROC_AUC_Macro": float(np.nanmean(roc_aucs)) if y_prob is not None else np.nan,
        "PR_AUC_Macro": float(np.nanmean(pr_aucs)) if y_prob is not None else np.nan,
        "MEL_Recall": float(tprs[MEL_IDX]),
        "MEL_Precision": float(ppvs[MEL_IDX]),
        "MEL_F1": float(f1s[MEL_IDX]),
        "MEL_Specificity": float(tnrs[MEL_IDX]),
    }
    return summary, df_per_class, cm

# --- 3.3 Decision Threshold Optimization (Priority 1.2) ---
def predict_with_thresholds(y_prob, thresholds):
    """
    Deterministic multiclass decision rule (Eq. 1 in manuscript):
        y_hat = argmax_k (p_k / theta_k)
    where theta_k in (0, 1) is the class-specific decision threshold calibrated on the validation set.
    """
    thetas = np.asarray(thresholds, dtype=np.float32).reshape(1, -1)
    scaled_scores = y_prob / np.clip(thetas, 1e-4, 1.0)
    return np.argmax(scaled_scores, axis=1)

def optimize_class_thresholds(y_val, y_val_prob, grid=np.arange(0.10, 0.91, 0.05), n_passes=3):
    """
    Calibrates per-class thresholds theta_k in [0.10, 0.90] strictly on the Validation Set
    via coordinate ascent to maximize a clinically oriented objective:
        J(theta) = 0.45 * F1_Macro + 0.35 * Recall_Macro + 0.20 * Recall_Melanoma
    """
    best_thetas = np.full(N_CLASSES, 0.50, dtype=np.float32)
    
    def objective_fn(thetas):
        preds = predict_with_thresholds(y_val_prob, thetas)
        f1_m = f1_score(y_val, preds, average="macro", zero_division=0)
        rec_m = recall_score(y_val, preds, average="macro", zero_division=0)
        rec_mel = recall_score(y_val, preds, labels=[MEL_IDX], average="micro", zero_division=0)
        return 0.45 * f1_m + 0.35 * rec_m + 0.20 * rec_mel

    best_score = objective_fn(best_thetas)
    for _ in range(n_passes):
        for k in range(N_CLASSES):
            cur_t = best_thetas[k]
            for cand in grid:
                best_thetas[k] = cand
                sc = objective_fn(best_thetas)
                if sc > best_score + 1e-6:
                    best_score = sc
                    cur_t = cand
            best_thetas[k] = cur_t
    return best_thetas

# --- 3.4 Probabilistic Calibration: Multiclass Brier Score, ECE & Temperature Scaling (Priority 1.6) ---
def compute_calibration_metrics(y_true, y_prob, n_bins=10):
    """
    Computes Multiclass Brier Score and Expected Calibration Error (ECE) across 10 equal-width bins.
    """
    y_onehot = np.eye(N_CLASSES)[y_true]
    brier_multi = float(np.mean(np.sum((y_prob - y_onehot) ** 2, axis=1)))
    
    confidences = np.max(y_prob, axis=1)
    predictions = np.argmax(y_prob, axis=1)
    accuracies = (predictions == y_true).astype(np.float32)
    
    bin_boundaries = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    bin_stats = []
    for b in range(n_bins):
        in_bin = (confidences > bin_boundaries[b]) & (confidences <= bin_boundaries[b + 1])
        prop_in_bin = np.mean(in_bin)
        if prop_in_bin > 0:
            acc_in_bin = float(np.mean(accuracies[in_bin]))
            conf_in_bin = float(np.mean(confidences[in_bin]))
            ece += np.abs(acc_in_bin - conf_in_bin) * prop_in_bin
            bin_stats.append({
                "bin_mid": 0.5 * (bin_boundaries[b] + bin_boundaries[b + 1]),
                "acc": acc_in_bin,
                "conf": conf_in_bin,
                "count": int(np.sum(in_bin)),
            })
    return {"Brier_Score": brier_multi, "ECE": float(ece), "bins": bin_stats}

def fit_temperature_scaling(y_val, y_val_prob):
    """Finds optimal temperature T > 0 on validation log-probabilities to minimize NLL."""
    log_p = np.log(np.clip(y_val_prob, 1e-7, 1.0))
    best_T, best_nll = 1.0, np.inf
    for T in np.linspace(0.5, 3.0, 101):
        scaled = log_p / T
        scaled -= np.max(scaled, axis=1, keepdims=True)
        probs_T = np.exp(scaled) / np.sum(np.exp(scaled), axis=1, keepdims=True)
        nll = -np.mean(np.log(np.clip(probs_T[np.arange(len(y_val)), y_val], 1e-7, 1.0)))
        if nll < best_nll:
            best_nll = nll
            best_T = float(T)
    return best_T

def apply_temperature_scaling(y_prob, T):
    log_p = np.log(np.clip(y_prob, 1e-7, 1.0)) / T
    log_p -= np.max(log_p, axis=1, keepdims=True)
    return np.exp(log_p) / np.sum(np.exp(log_p), axis=1, keepdims=True)

# --- 3.5 Grouped Bootstrap by lesion_id for 95% CIs (Priority 1.5) ---
def grouped_bootstrap_ci(y_true, y_pred, y_prob, groups, n_boot=N_BOOTSTRAP, seed=MASTER_SEED):
    """
    Cluster bootstrap resampling at the lesion_id level.
    All images belonging to a sampled lesion_id are drawn together.
    """
    rng = np.random.default_rng(seed)
    unique_groups = np.unique(groups)
    n_groups = len(unique_groups)
    
    # Map each group to its sample indices for fast cluster resampling
    group_to_indices = {g: np.where(groups == g)[0] for g in unique_groups}
    
    boot_records = []
    for _ in range(n_boot):
        sampled_groups = rng.choice(unique_groups, size=n_groups, replace=True)
        idx = np.concatenate([group_to_indices[g] for g in sampled_groups])
        yb_t = y_true[idx]
        yb_p = y_pred[idx]
        
        cm = confusion_matrix(yb_t, yb_p, labels=np.arange(N_CLASSES))
        N = cm.sum()
        tp = np.diag(cm).astype(np.float64)
        fn = cm.sum(axis=1) - tp
        fp = cm.sum(axis=0) - tp
        tn = N - (tp + fn + fp)
        
        tpr = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) > 0)
        tnr = np.divide(tn, tn + fp, out=np.zeros_like(tn), where=(tn + fp) > 0)
        ppv = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
        f1 = np.divide(2 * tp, 2 * tp + fp + fn, out=np.zeros_like(tp), where=(2 * tp + fp + fn) > 0)
        acc_ovr = (tp + tn) / N
        
        den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
        mcc_k = np.divide(tp * tn - fp * fn, den, out=np.zeros_like(tp), where=den > 0)
        
        boot_records.append({
            "Acc_Multi": float(tp.sum() / N),
            "Acc_Mean_OvR": float(np.mean(acc_ovr)),
            "BA_Multi_Recall_Macro": float(np.mean(tpr)),
            "Precision_Macro": float(np.mean(ppv)),
            "Specificity_Macro": float(np.mean(tnr)),
            "F1_Macro": float(np.mean(f1)),
            "MCC_Macro_OvR": float(np.mean(mcc_k)),
            "MEL_Recall": float(tpr[MEL_IDX]),
            "MEL_F1": float(f1[MEL_IDX]),
        })
        
    df_boot = pd.DataFrame(boot_records)
    ci_summary = {}
    for col in df_boot.columns:
        ci_summary[col] = {
            "mean": float(df_boot[col].mean()),
            "std": float(df_boot[col].std()),
            "ci_low": float(np.percentile(df_boot[col], 2.5)),
            "ci_high": float(np.percentile(df_boot[col], 97.5)),
        }
    return ci_summary

# --- 3.6 Paired Statistical Comparison: McNemar & Paired Cluster Bootstrap (Priority 2.5) ---
def paired_statistical_comparison(y_true, pred_a, pred_b, groups, n_boot=N_BOOTSTRAP, seed=MASTER_SEED):
    """
    Compares Model A vs. Model B on paired test predictions using:
    1. McNemar's test with continuity correction on exact multiclass correctness.
    2. Paired Cluster Bootstrap (by lesion_id) for Delta Recall_Macro, Delta F1_Macro, and Delta Acc_Multi.
    """
    from scipy.stats import chi2
    correct_a = (pred_a == y_true)
    correct_b = (pred_b == y_true)
    
    # Discordant pairs: b01 (A right, B wrong), b10 (A wrong, B right)
    b01 = int(np.sum(correct_a & ~correct_b))
    b10 = int(np.sum(~correct_a & correct_b))
    if (b01 + b10) > 0:
        chi2_stat = ((abs(b01 - b10) - 1.0) ** 2) / (b01 + b10)
        p_mcnemar = float(1.0 - chi2.cdf(chi2_stat, df=1))
    else:
        chi2_stat, p_mcnemar = 0.0, 1.0
        
    rng = np.random.default_rng(seed)
    unique_groups = np.unique(groups)
    group_to_indices = {g: np.where(groups == g)[0] for g in unique_groups}
    
    delta_acc, delta_rec, delta_f1 = [], [], []
    for _ in range(n_boot):
        sampled_groups = rng.choice(unique_groups, size=len(unique_groups), replace=True)
        idx = np.concatenate([group_to_indices[g] for g in sampled_groups])
        yt, pa, pb = y_true[idx], pred_a[idx], pred_b[idx]
        delta_acc.append(accuracy_score(yt, pa) - accuracy_score(yt, pb))
        delta_rec.append(recall_score(yt, pa, average="macro", zero_division=0) - recall_score(yt, pb, average="macro", zero_division=0))
        delta_f1.append(f1_score(yt, pa, average="macro", zero_division=0) - f1_score(yt, pb, average="macro", zero_division=0))
        
    delta_rec = np.array(delta_rec)
    p_boot_two_sided = float(2.0 * min(np.mean(delta_rec <= 0), np.mean(delta_rec >= 0)))
    return {
        "b01_A_only": b01,
        "b10_B_only": b10,
        "McNemar_chi2": chi2_stat,
        "McNemar_p": p_mcnemar,
        "Delta_Acc_Multi_mean": float(np.mean(delta_acc)),
        "Delta_Acc_Multi_CI95": [float(np.percentile(delta_acc, 2.5)), float(np.percentile(delta_acc, 97.5))],
        "Delta_Recall_Macro_mean": float(np.mean(delta_rec)),
        "Delta_Recall_Macro_CI95": [float(np.percentile(delta_rec, 2.5)), float(np.percentile(delta_rec, 97.5))],
        "Delta_F1_Macro_mean": float(np.mean(delta_f1)),
        "Delta_F1_Macro_CI95": [float(np.percentile(delta_f1, 2.5)), float(np.percentile(delta_f1, 97.5))],
        "Paired_Boot_p": p_boot_two_sided,
    }

## 4. Random Forest Baseline: Progressive Feature Journey & Ablation Study (Priority 1 & 5)
We evaluate the Random Forest classifier under the exact same lesion-grouped splits (`train_idx`, `val_idx`, `test_idx`):
1. **Majority Class Baseline (`NV`):** Theoretical & empirical lower bound.
2. **Progressive Feature Addition:**
   - *Stage A:* Color Only (RGB + HSV means: 6 features)
   - *Stage B:* Color + Texture (GLCM + LBP: 23 features)
   - *Stage C:* Full ABCD + DullRazor (Color + Texture + Otsu/Hu Moments: 30 features)
3. **Component Ablation Study:** Isolating the effect of DullRazor hair removal and the 7 Hu shape descriptors.
4. **Threshold-Optimized Random Forest (`RF Optimized`):** Applying validation-calibrated thresholds $\theta_k$ to the full RF baseline.

In [ ]:
all_stage_summaries = {}
all_stage_per_class = {}
computational_costs = []

# --- 4.1 Majority Classifier Baseline (Predicts NV for all test samples) ---
y_test = y_enc[test_idx]
y_val = y_enc[val_idx]
y_train = y_enc[train_idx]
test_lesions = lesion_ids[test_idx]

y_pred_maj = np.full_like(y_test, fill_value=NV_IDX)
y_prob_maj = np.zeros((len(y_test), N_CLASSES), dtype=np.float32)
y_prob_maj[:, NV_IDX] = 1.0
maj_sum, maj_df, _ = compute_complete_metrics(y_test, y_pred_maj, y_prob_maj)
all_stage_summaries["Majority Classifier (NV)"] = maj_sum

# --- Helper to train and evaluate Random Forest configurations ---
def evaluate_rf_experiment(X_feature_matrix, stage_name, use_smoteenn=False):
    t0 = time.time()
    sc = StandardScaler()
    X_tr = sc.fit_transform(X_feature_matrix[train_idx])
    X_va = sc.transform(X_feature_matrix[val_idx])
    X_te = sc.transform(X_feature_matrix[test_idx])
    
    if use_smoteenn:
        sme = SMOTEENN(random_state=MASTER_SEED)
        X_tr_fit, y_tr_fit = sme.fit_resample(X_tr, y_train)
    else:
        X_tr_fit, y_tr_fit = X_tr, y_train
        
    rf = RandomForestClassifier(
        n_estimators=500,
        max_depth=20,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=MASTER_SEED,
        n_jobs=-1,
    )
    rf.fit(X_tr_fit, y_tr_fit)
    train_time_s = time.time() - t0
    
    t_inf = time.time()
    y_prob_te = rf.predict_proba(X_te)
    inf_ms = ((time.time() - t_inf) / len(X_te)) * 1000.0 + feat_time_per_img_ms
    y_pred_te = np.argmax(y_prob_te, axis=1)
    
    summary, df_cls, cm = compute_complete_metrics(y_test, y_pred_te, y_prob_te)
    summary["Train_Time_s"] = round(train_time_s, 2)
    summary["Inf_Latency_ms"] = round(inf_ms, 2)
    return rf, sc, summary, df_cls, cm, y_prob_te, y_pred_te, X_va

# Feature slices: Color (0:6), Texture (6:23), Shape Hu (23:30)
print("--- Running Progressive Random Forest Experiments & Ablation Study ---")
_, _, sum_color, _, _, _, _, _ = evaluate_rf_experiment(X_feat_dr[:, :6], "RF Stage 1: Color Only (HSV/RGB)")
_, _, sum_col_tex_dr, _, _, _, _, _ = evaluate_rf_experiment(X_feat_dr[:, :23], "RF Stage 2: Color + Texture (with DullRazor)")
_, _, sum_col_tex_nodr, _, _, _, _, _ = evaluate_rf_experiment(X_feat_nodr[:, :23], "Ablation: Color + Texture (no DullRazor, no Shape)")
_, _, sum_full_nodr, _, _, _, _, _ = evaluate_rf_experiment(X_feat_nodr, "Ablation: Color + Texture + Shape (no DullRazor)")
rf_full, sc_rf, sum_rf_full, df_rf_full, cm_rf_full, prob_rf_test, pred_rf_test, X_va_rf = evaluate_rf_experiment(
    X_feat_dr, "Baseline RF Complete (ABCD + DullRazor)"
)
_, _, sum_rf_smote, _, _, _, _, _ = evaluate_rf_experiment(
    X_feat_dr, "Baseline RF + SMOTEENN", use_smoteenn=True
)

# Threshold-Optimized Random Forest (calibrated strictly on Validation Set)
prob_rf_val = rf_full.predict_proba(X_va_rf)
thetas_rf = optimize_class_thresholds(y_val, prob_rf_val)
pred_rf_opt = predict_with_thresholds(prob_rf_test, thetas_rf)
sum_rf_opt, df_rf_opt, cm_rf_opt = compute_complete_metrics(y_test, pred_rf_opt, prob_rf_test)

all_stage_summaries["RF Stage 1: Color Only"] = sum_color
all_stage_summaries["RF Stage 2: Color + Texture (No DR)"] = sum_col_tex_nodr
all_stage_summaries["RF Stage 2: Color + Texture (With DR)"] = sum_col_tex_dr
all_stage_summaries["RF Ablation: Full ABCD (No DR)"] = sum_full_nodr
all_stage_summaries["Baseline RF Complete"] = sum_rf_full
all_stage_summaries["Baseline RF + SMOTEENN"] = sum_rf_smote
all_stage_summaries["Baseline RF Optimized"] = sum_rf_opt
all_stage_per_class["Baseline RF Complete"] = df_rf_full

computational_costs.append({
    "Model": "Random Forest (500 trees + ABCD)",
    "Total_Params": "N/A (Tree Ensemble)",
    "Trainable_Params": "N/A",
    "Train_Time_s": sum_rf_full["Train_Time_s"],
    "Epoch_Time_s": sum_rf_full["Train_Time_s"],
    "Latency_ms_per_img": sum_rf_full["Inf_Latency_ms"],
    "Peak_VRAM_MB": 0.0,
})

df_rf_ablation = pd.DataFrame([
    {"Configuration": k, **v}
    for k, v in all_stage_summaries.items()
    if "RF" in k or "Majority" in k
])
print("\n--- Progressive RF & Ablation Summary (Test Set) ---")
display_cols = ["Configuration", "Acc_Multi", "Acc_Mean_OvR", "BA_Multi_Recall_Macro", "Precision_Macro", "F1_Macro", "MCC_Macro_OvR", "ROC_AUC_Macro", "MEL_Recall"]
print(df_rf_ablation[display_cols].to_string(index=False))
df_rf_ablation.to_csv(ARTIFACT_DIR / "rf_progression_and_ablation.csv", index=False)

## 5. Comparative Deep Learning Architectures & Multi-Seed Protocol (Priorities 1 & 2)

To satisfy **Priority 1.1** (exact correspondence between code and text) and **Priority 2** (ResNet50 and ConvNeXt baselines, multi-seed evaluation, paired statistical testing, and computational cost comparison):
- **Important Preprocessing Fix:** `tf.keras.applications.EfficientNetV2B0`, `ConvNeXtTiny`, and `ResNet50` are initialized with their native preprocessing layers (`include_preprocessing=True` for EfficientNetV2/ConvNeXt expecting `[0, 255]` float32 tensors, and `tf.keras.applications.resnet50.preprocess_input` for ResNet50), avoiding the double `/255.0` division bug.
- **Architectures Evaluated:**
  1. `ResNet50` (Baseline CNN)
  2. `ConvNeXtTiny` (Modernized ConvNet Baseline)
  3. `EfficientNetV2B0_Visual` (Visual-Only Backbone, isolating the deep representation contribution)
  4. `EfficientNetV2B0_Hybrid` (Proposed Hybrid Architecture fusing `EfficientNetV2B0` visual features + 30 tabular `ABCD` clinical features)
- **Multi-Seed Training (`SEEDS = [42, 100, 2024]`):** Each architecture is trained across multiple random seeds under identical data partitions, Two-Stage Fine-Tuning, Sparse Focal Loss ($\gamma=2.0$), and balanced dataset sampling.

In [ ]:
# --- 5.1 Sparse Focal Loss & Optimizer Builder ---
def sparse_focal_loss(gamma=2.0, alpha=None):
    gamma = float(gamma)
    def loss_fn(y_true, y_pred):
        y_true = tf.reshape(tf.cast(y_true, tf.int32), [-1])
        y_pred = tf.cast(y_pred, tf.float32)
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1.0 - tf.keras.backend.epsilon())
        p_t = tf.gather(y_pred, y_true, axis=1, batch_dims=1)
        loss = tf.pow(1.0 - p_t, gamma) * -tf.math.log(p_t)
        if alpha is not None:
            alpha_t = tf.convert_to_tensor(alpha, dtype=tf.float32)
            loss = loss * (alpha_t if alpha_t.shape.rank == 0 else tf.gather(alpha_t, y_true))
        return tf.reduce_mean(loss)
    return loss_fn

def build_adam_optimizer(learning_rate, accum_steps=ACCUM_STEPS):
    if accum_steps > 1:
        try:
            return optimizers.AdamW(learning_rate=learning_rate, weight_decay=1e-4, gradient_accumulation_steps=accum_steps)
        except (TypeError, ValueError):
            return optimizers.Adam(learning_rate=learning_rate)
    try:
        return optimizers.AdamW(learning_rate=learning_rate, weight_decay=1e-4)
    except Exception:
        return optimizers.Adam(learning_rate=learning_rate)

# --- 5.2 GPU-Compiled Data Augmentation & tf.data Pipelines ---
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.15),
    layers.RandomTranslation(height_factor=0.08, width_factor=0.08),
], name="dermatoscopic_augmentation")

# Keep images in [0, 255] float32 so backbone-specific preprocessing handles normalization accurately
def make_tf_datasets(X_img, X_tab, y_arr, tr_idx, va_idx, te_idx, seed=MASTER_SEED):
    X_tr_i, f_tr, y_tr = X_img[tr_idx], X_tab[tr_idx], y_arr[tr_idx]
    X_va_i, f_va, y_va = X_img[va_idx], X_tab[va_idx], y_arr[va_idx]
    X_te_i, f_te, y_te = X_img[te_idx], X_tab[te_idx], y_arr[te_idx]

    def map_fn(img_u8, tab_f, lbl):
        rgb_255 = tf.cast(img_u8, tf.float32)  # [0, 255] range
        return {"rgb_input": rgb_255, "abcd_input": tab_f}, lbl

    class_datasets = []
    for cls in range(N_CLASSES):
        mask = (y_tr == cls)
        ds_c = tf.data.Dataset.from_tensor_slices((X_tr_i[mask], f_tr[mask], y_tr[mask]))
        ds_c = ds_c.shuffle(buffer_size=min(int(np.sum(mask)), 1000), seed=seed).repeat()
        class_datasets.append(ds_c)

    steps_per_epoch = len(y_tr) // BATCH_SIZE
    train_ds = (
        _sample_from_datasets(class_datasets, weights=[1.0 / N_CLASSES] * N_CLASSES, seed=seed)
        .map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(BATCH_SIZE, drop_remainder=True)
        .prefetch(tf.data.AUTOTUNE)
    )
    val_ds = (
        tf.data.Dataset.from_tensor_slices((X_va_i, f_va, y_va))
        .map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )
    test_ds = (
        tf.data.Dataset.from_tensor_slices((X_te_i, f_te, y_te))
        .map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )
    return train_ds, val_ds, test_ds, steps_per_epoch

# --- 5.3 Model Factory for Comparative Architectures (Priority 2.1, 2.2, 2.4) ---
def build_comparative_model(arch_name, n_tabular=N_TABULAR_FEATURES, use_hybrid=True):
    """
    Builds one of:
      - 'EfficientNetV2B0_Hybrid' (EfficientNetV2B0 + ABCD Tabular Fusion)
      - 'EfficientNetV2B0_Visual' (EfficientNetV2B0 Image-Only)
      - 'ResNet50'                (ResNet50 Baseline)
      - 'ConvNeXtTiny'            (ConvNeXt-Tiny Baseline)
    Returns (model, backbone).
    """
    in_rgb = layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3), name="rgb_input")
    in_abcd = layers.Input(shape=(n_tabular,), name="abcd_input")
    x_aug = data_augmentation(in_rgb)
    
    if "EfficientNetV2B0" in arch_name:
        backbone = tf.keras.applications.EfficientNetV2B0(
            include_top=False, weights="imagenet", input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), include_preprocessing=True
        )
        x_vis = backbone(x_aug)
    elif arch_name == "ResNet50":
        # ResNet50 expects caffe-style zero-centered BGR preprocessing on [0, 255]
        x_prep = layers.Lambda(lambda z: tf.keras.applications.resnet50.preprocess_input(tf.cast(z, tf.float32)), name="resnet_prep")(x_aug)
        backbone = tf.keras.applications.ResNet50(
            include_top=False, weights="imagenet", input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
        )
        x_vis = backbone(x_prep)
    elif arch_name == "ConvNeXtTiny":
        try:
            backbone = tf.keras.applications.ConvNeXtTiny(
                include_top=False, weights="imagenet", input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), include_preprocessing=True
            )
            x_vis = backbone(x_aug)
        except Exception:
            # Fallback to DenseNet121 if ConvNeXtTiny is unavailable in older TF build
            x_prep = layers.Lambda(lambda z: tf.keras.applications.densenet.preprocess_input(tf.cast(z, tf.float32)), name="densenet_prep")(x_aug)
            backbone = tf.keras.applications.DenseNet121(
                include_top=False, weights="imagenet", input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
            )
            x_vis = backbone(x_prep)
    else:
        raise ValueError(f"Unknown architecture: {arch_name}")

    backbone.trainable = False  # Phase 1: Warmup with frozen backbone
    x1 = layers.GlobalAveragePooling2D(name="gap")(x_vis)
    x1 = layers.Dropout(0.3, name="vis_dropout")(x1)
    x1 = layers.Dense(128, activation="relu", name="vis_dense")(x1)

    if use_hybrid:
        x2 = layers.Dense(64, activation="relu", name="tab_dense")(in_abcd)
        comb = layers.Concatenate(name="hybrid_concat")([x1, x2])
    else:
        comb = x1

    comb = layers.Dropout(0.4, name="head_dropout")(comb)
    out = layers.Dense(N_CLASSES, activation="softmax", dtype="float32", name="Classifier_Softmax")(comb)
    model = models.Model(inputs=[in_rgb, in_abcd], outputs=out, name=arch_name)
    return model, backbone

def unfreeze_top_layers(backbone, fraction=0.35):
    """Unfreezes the top fraction of layers in the backbone while keeping BatchNormalization frozen."""
    backbone.trainable = True
    n_layers = len(backbone.layers)
    cutoff = int(n_layers * (1.0 - fraction))
    for i, layer in enumerate(backbone.layers):
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
        elif i >= cutoff:
            layer.trainable = True
        else:
            layer.trainable = False

# Build datasets once for the full feature tensor
X_tab_all = np.zeros((len(y_enc), N_TABULAR_FEATURES), dtype=np.float32)
X_tab_all[train_idx] = f_train_scaled
X_tab_all[val_idx] = f_val_scaled
X_tab_all[test_idx] = f_test_scaled

def train_and_evaluate_dl_model(arch_name, use_hybrid, seed=MASTER_SEED, warmup_ep=WARMUP_EPOCHS, finetune_ep=FINETUNE_EPOCHS):
    set_global_seed(seed)
    tf.keras.backend.clear_session()
    gc.collect()
    
    train_ds, val_ds, test_ds, steps_per_epoch = make_tf_datasets(
        X_raw_clean, X_tab_all, y_enc, train_idx, val_idx, test_idx, seed=seed
    )
    
    with strategy.scope():
        model, backbone = build_comparative_model(arch_name, use_hybrid=use_hybrid)
        total_params = model.count_params()
        model.compile(
            optimizer=build_adam_optimizer(1e-4, ACCUM_STEPS),
            loss=sparse_focal_loss(gamma=2.0),
            metrics=["accuracy"],
        )
        
    print(f"\n=================================================================")
    print(f"Training {arch_name} (Hybrid={use_hybrid}, Seed={seed}) | Params={total_params:,}")
    print(f"=================================================================")
    
    t_start = time.time()
    # Phase 1: Head Warmup
    cb_warmup = [callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=0)]
    h1 = model.fit(train_ds, steps_per_epoch=steps_per_epoch, validation_data=val_ds, epochs=warmup_ep, callbacks=cb_warmup, verbose=1)
    
    # Phase 2: Fine-Tuning Top Backbone Blocks
    with strategy.scope():
        unfreeze_top_layers(backbone, fraction=0.35)
        trainable_params = int(np.sum([tf.keras.backend.count_params(w) for w in model.trainable_weights]))
        model.compile(
            optimizer=build_adam_optimizer(1e-5, ACCUM_STEPS),
            loss=sparse_focal_loss(gamma=2.0),
            metrics=["accuracy"],
        )
        
    cb_ft = [
        callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, min_lr=1e-7, verbose=0),
    ]
    h2 = model.fit(train_ds, steps_per_epoch=steps_per_epoch, validation_data=val_ds, epochs=finetune_ep, callbacks=cb_ft, verbose=1)
    total_train_time = time.time() - t_start
    epochs_ran = len(h1.history["loss"]) + len(h2.history["loss"])
    epoch_time = total_train_time / max(1, epochs_ran)
    
    # Validation & Test Inference (strictly isolated)
    val_probs = model.predict(val_ds, verbose=0)
    t_inf = time.time()
    test_probs = model.predict(test_ds, verbose=0)
    inf_latency_ms = ((time.time() - t_inf) / len(test_idx)) * 1000.0 + (feat_time_per_img_ms if use_hybrid else 0.0)
    
    # Uncalibrated argmax predictions
    test_preds_raw = np.argmax(test_probs, axis=1)
    sum_raw, df_cls_raw, cm_raw = compute_complete_metrics(y_test, test_preds_raw, test_probs)
    
    # Threshold Optimization strictly on Validation Set
    opt_thetas = optimize_class_thresholds(y_val, val_probs)
    test_preds_opt = predict_with_thresholds(test_probs, opt_thetas)
    sum_opt, df_cls_opt, cm_opt = compute_complete_metrics(y_test, test_preds_opt, test_probs)
    
    mem_info = get_gpu_memory_mb()
    cost_info = {
        "Model": arch_name,
        "Seed": seed,
        "Total_Params": int(total_params),
        "Trainable_Params": int(trainable_params),
        "Train_Time_s": round(total_train_time, 2),
        "Epoch_Time_s": round(epoch_time, 2),
        "Latency_ms_per_img": round(inf_latency_ms, 2),
        "Peak_VRAM_MB": mem_info["peak_mb"],
    }
    return {
        "model": model,
        "backbone": backbone,
        "val_probs": val_probs,
        "test_probs": test_probs,
        "test_preds_raw": test_preds_raw,
        "test_preds_opt": test_preds_opt,
        "opt_thetas": opt_thetas,
        "sum_raw": sum_raw,
        "sum_opt": sum_opt,
        "df_cls_raw": df_cls_raw,
        "df_cls_opt": df_cls_opt,
        "cm_raw": cm_raw,
        "cm_opt": cm_opt,
        "cost": cost_info,
    }

## 6. Executing Multi-Architecture & Multi-Seed Benchmarks (Priority 2)
We execute:
1. **Multi-Seed Evaluation (`SEEDS = [42, 100, 2024]`)** across `ResNet50`, `ConvNeXtTiny`, `EfficientNetV2B0_Visual`, and `EfficientNetV2B0_Hybrid` to report **Mean $\pm$ SD**.
2. **Retention of Primary Models (`seed = 42`)** for paired statistical hypothesis tests (McNemar & Paired Cluster Bootstrap), Grad-CAM explainability, calibration curves, and melanoma error auditing.

In [ ]:
ARCHITECTURES = [
    ("ResNet50", False),
    ("ConvNeXtTiny", False),
    ("EfficientNetV2B0_Visual", False),
    ("EfficientNetV2B0_Hybrid", True),
]

primary_runs = {}       # Stores seed=42 full artifacts for each architecture
multi_seed_records = [] # Stores metrics across all seeds [42, 100, 2024]

for arch_name, is_hybrid in ARCHITECTURES:
    for s in MULTI_SEEDS:
        res = train_and_evaluate_dl_model(arch_name, use_hybrid=is_hybrid, seed=s)
        
        rec_raw = {"Architecture": arch_name, "Variant": "Standard (Argmax)", "Seed": s, **res["sum_raw"], **res["cost"]}
        rec_opt = {"Architecture": arch_name, "Variant": "Threshold-Optimized", "Seed": s, **res["sum_opt"], **res["cost"]}
        multi_seed_records.append(rec_raw)
        multi_seed_records.append(rec_opt)
        
        if s == MASTER_SEED:
            primary_runs[arch_name] = res
            computational_costs.append(res["cost"])
            # Save primary model weights only for seed=42 to conserve memory
            res["model"].save_weights(str(ARTIFACT_DIR / f"{arch_name}_seed{s}.weights.h5"))
        else:
            # Free memory from secondary seed runs immediately
            del res["model"], res["backbone"]
            gc.collect()

df_multi_seed = pd.DataFrame(multi_seed_records)
df_multi_seed.to_csv(ARTIFACT_DIR / "multi_seed_all_runs.csv", index=False)

# Aggregate Mean +/- SD across seeds (Priority 2.3)
metric_cols = ["Acc_Multi", "Acc_Mean_OvR", "BA_Multi_Recall_Macro", "BA_Mean_OvR", "Precision_Macro", "Specificity_Macro", "F1_Macro", "MCC_Macro_OvR", "ROC_AUC_Macro", "PR_AUC_Macro", "MEL_Recall"]
df_seed_agg = df_multi_seed.groupby(["Architecture", "Variant"])[metric_cols].agg(["mean", "std"]).reset_index()
df_seed_agg.to_csv(ARTIFACT_DIR / "multi_seed_summary_mean_sd.csv", index=False)
print("\n--- Multi-Seed Evaluation Summary (Mean +/- SD across seeds) ---")
print(df_seed_agg.to_string(index=False))

# Register primary EfficientNetV2B0 runs into main progression table
eff_res = primary_runs["EfficientNetV2B0_Hybrid"]
all_stage_summaries["ResNet50 Baseline"] = primary_runs["ResNet50"]["sum_raw"]
all_stage_summaries["ConvNeXt-Tiny Baseline"] = primary_runs["ConvNeXtTiny"]["sum_raw"]
all_stage_summaries["DL Visual-Only (EfficientNetV2B0)"] = primary_runs["EfficientNetV2B0_Visual"]["sum_raw"]
all_stage_summaries["DL Complete (EfficientNetV2B0 Hybrid)"] = eff_res["sum_raw"]
all_stage_summaries["DL Optimized (EfficientNetV2B0 Hybrid)"] = eff_res["sum_opt"]

df_progression = pd.DataFrame([{"Stage_Model": k, **v} for k, v in all_stage_summaries.items()])
df_progression.to_csv(ARTIFACT_DIR / "table1_progression_metrics.csv", index=False)

df_costs = pd.DataFrame(computational_costs)
df_costs.to_csv(ARTIFACT_DIR / "table_computational_costs.csv", index=False)
print("\n--- Computational Cost Comparison (Priority 2.6) ---")
print(df_costs.to_string(index=False))

## 7. Statistical Uncertainty (Grouped Bootstrap), Probabilistic Calibration & Paired Hypothesis Tests (Priority 1.5, 1.6 & 2.5)
We compute:
1. **1,000-iteration Lesion-Grouped Bootstrap 95% Confidence Intervals** for the flagship `DL Optimized (EfficientNetV2B0 Hybrid)` model.
2. **Probabilistic Calibration Analysis:** Multiclass Brier Score, Expected Calibration Error (ECE, 10 bins), Temperature Scaling ($T$), and a **Reliability Diagram**.
3. **Paired Statistical Comparisons:** McNemar's $\chi^2$ test and Paired Cluster Bootstrap between `EfficientNetV2B0_Hybrid` and all baselines (`Random Forest`, `ResNet50`, `ConvNeXtTiny`, `EfficientNetV2B0_Visual`).

In [ ]:
# --- 7.1 Grouped Bootstrap 95% CIs for Flagship Model ---
print("Running 1,000-iteration Grouped Bootstrap by lesion_id...")
ci_eff_opt = grouped_bootstrap_ci(
    y_test, eff_res["test_preds_opt"], eff_res["test_probs"], test_lesions, n_boot=N_BOOTSTRAP
)
with open(ARTIFACT_DIR / "grouped_bootstrap_ci95.json", "w") as f:
    json.dump(ci_eff_opt, f, indent=2)

print("\n--- Grouped Bootstrap 95% Confidence Intervals (DL Optimized) ---")
for metric_name, stats in ci_eff_opt.items():
    print(f"  {metric_name:24s}: {stats['mean']:.3f} (95% CI: {stats['ci_low']:.3f} -- {stats['ci_high']:.3f})")

# --- 7.2 Probabilistic Calibration & Reliability Diagram (Priority 1.6) ---
cal_raw = compute_calibration_metrics(y_test, eff_res["test_probs"], n_bins=10)
opt_T = fit_temperature_scaling(y_val, eff_res["val_probs"])
test_probs_temp = apply_temperature_scaling(eff_res["test_probs"], opt_T)
cal_temp = compute_calibration_metrics(y_test, test_probs_temp, n_bins=10)

print(f"\n--- Probabilistic Calibration Audit ---")
print(f"  Uncalibrated Probabilities : Brier Score = {cal_raw['Brier_Score']:.4f} | ECE = {cal_raw['ECE']:.4f} ({cal_raw['ECE']*100:.2f}%)")
print(f"  Temperature Scaled (T={opt_T:.2f}) : Brier Score = {cal_temp['Brier_Score']:.4f} | ECE = {cal_temp['ECE']:.4f} ({cal_temp['ECE']*100:.2f}%)")
print(f"  Optimal Class Thresholds theta_k : {dict(zip(CLASS_CODES, np.round(eff_res['opt_thetas'], 3)))}")

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.plot([0, 1], [0, 1], "k--", label="Perfect Calibration")
bins_raw_df = pd.DataFrame(cal_raw["bins"])
bins_temp_df = pd.DataFrame(cal_temp["bins"])
if not bins_raw_df.empty:
    ax.plot(bins_raw_df["conf"], bins_raw_df["acc"], "o-", lw=2, label=f"Raw Softmax (ECE={cal_raw['ECE']:.3f}, Brier={cal_raw['Brier_Score']:.3f})")
if not bins_temp_df.empty:
    ax.plot(bins_temp_df["conf"], bins_temp_df["acc"], "s-", lw=2, label=f"Temp. Scaled T={opt_T:.2f} (ECE={cal_temp['ECE']:.3f}, Brier={cal_temp['Brier_Score']:.3f})")
ax.set_xlabel("Mean Predicted Confidence")
ax.set_ylabel("Empirical Fraction of Positives (Accuracy)")
ax.set_title("Reliability Diagram (Probabilistic Calibration on Test Set)")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "calibration_reliability.pdf")
plt.savefig(ARTIFACT_DIR / "calibration_reliability.png")
plt.show()

# --- 7.3 Paired Statistical Hypothesis Tests (Priority 2.5) ---
paired_comparisons = []
pred_flagship = eff_res["test_preds_opt"]
comparison_targets = [
    ("Random Forest Complete", pred_rf_test),
    ("ResNet50", primary_runs["ResNet50"]["test_preds_raw"]),
    ("ConvNeXtTiny", primary_runs["ConvNeXtTiny"]["test_preds_raw"]),
    ("EfficientNetV2B0 (Visual-Only)", primary_runs["EfficientNetV2B0_Visual"]["test_preds_raw"]),
    ("EfficientNetV2B0 Hybrid (Pre-Threshold)", eff_res["test_preds_raw"]),
]
for comp_name, pred_other in comparison_targets:
    stat_res = paired_statistical_comparison(y_test, pred_flagship, pred_other, test_lesions, n_boot=N_BOOTSTRAP)
    paired_comparisons.append({"Comparison": f"EfficientNetV2B0_Hybrid_Opt vs. {comp_name}", **stat_res})

df_paired_stats = pd.DataFrame(paired_comparisons)
df_paired_stats.to_csv(ARTIFACT_DIR / "paired_statistical_tests.csv", index=False)
print("\n--- Paired Statistical Comparisons vs. Flagship Model ---")
print(df_paired_stats[["Comparison", "McNemar_chi2", "McNemar_p", "Delta_Acc_Multi_mean", "Delta_Recall_Macro_mean", "Delta_F1_Macro_mean", "Paired_Boot_p"]].to_string(index=False))

## 8. Generalization, Perturbation Robustness & Acquisition Domain Shift (Priority 3)
To address **Priority 3**:
1. **Perturbation Robustness Suite (Priority 3.3):** Evaluates `EfficientNetV2B0_Hybrid`, `ResNet50`, and `ConvNeXtTiny` on the test set under controlled photometric and acquisition corruptions:
   - Brightness shift ($\pm 20\%$), Contrast scaling ($0.75\times, 1.25\times$), Additive Gaussian noise ($\sigma = 15$), Optical Gaussian blur ($5 \times 5$), and JPEG compression artifacts ($Q = 40$).
2. **Acquisition & Population Sub-Distribution Shift (Priority 3.2):** Evaluates performance stratified by diagnostic confirmation modality (`dx_type`: histopathology vs. follow-up vs. expert consensus) and analyzes color/luminance distribution shifts across classes.

In [ ]:
def apply_perturbation_batch(X_imgs, kind, param):
    out = np.empty_like(X_imgs)
    for i, img in enumerate(X_imgs):
        if kind == "brightness":
            # param in [-50, 50]
            out[i] = np.clip(img.astype(np.int16) + int(param), 0, 255).astype(np.uint8)
        elif kind == "contrast":
            # param in [0.7, 1.3]
            mean_val = img.mean()
            out[i] = np.clip((img.astype(np.float32) - mean_val) * float(param) + mean_val, 0, 255).astype(np.uint8)
        elif kind == "gaussian_noise":
            rng = np.random.default_rng(MASTER_SEED + i)
            noise = rng.normal(0, float(param), img.shape)
            out[i] = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
        elif kind == "gaussian_blur":
            k = int(param)
            out[i] = cv2.GaussianBlur(img, (k, k), 0)
        elif kind == "jpeg_compression":
            encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), int(param)]
            _, encimg = cv2.imencode(".jpg", cv2.cvtColor(img, cv2.COLOR_RGB2BGR), encode_param)
            decimg = cv2.imdecode(encimg, 1)
            out[i] = cv2.cvtColor(decimg, cv2.COLOR_BGR2RGB)
        else:
            out[i] = img
    return out

PERTURBATIONS = [
    ("Clean (Unperturbed)", "none", 0),
    ("Brightness -20%", "brightness", -40),
    ("Brightness +20%", "brightness", +40),
    ("Contrast 0.75x", "contrast", 0.75),
    ("Contrast 1.25x", "contrast", 1.25),
    ("Gaussian Noise (sigma=15)", "gaussian_noise", 15.0),
    ("Optical Blur (5x5)", "gaussian_blur", 5),
    ("JPEG Compression (Q=40)", "jpeg_compression", 40),
]

X_test_imgs = X_raw_clean[test_idx]
robustness_records = []

print("--- Evaluating Perturbation Robustness Across Architectures (Priority 3.3) ---")
for pert_label, pert_kind, pert_param in PERTURBATIONS:
    X_pert = apply_perturbation_batch(X_test_imgs, pert_kind, pert_param)
    ds_pert = (
        tf.data.Dataset.from_tensor_slices(({"rgb_input": X_pert.astype(np.float32), "abcd_input": f_test_scaled}, y_test))
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )
    for arch_name in ["ResNet50", "ConvNeXtTiny", "EfficientNetV2B0_Hybrid"]:
        run_info = primary_runs[arch_name]
        probs_p = run_info["model"].predict(ds_pert, verbose=0)
        preds_p = predict_with_thresholds(probs_p, run_info["opt_thetas"]) if arch_name == "EfficientNetV2B0_Hybrid" else np.argmax(probs_p, axis=1)
        sum_p, _, _ = compute_complete_metrics(y_test, preds_p, probs_p)
        robustness_records.append({
            "Perturbation": pert_label,
            "Architecture": arch_name,
            "Acc_Multi": sum_p["Acc_Multi"],
            "BA_Multi_Recall_Macro": sum_p["BA_Multi_Recall_Macro"],
            "F1_Macro": sum_p["F1_Macro"],
            "MCC_Macro_OvR": sum_p["MCC_Macro_OvR"],
            "MEL_Recall": sum_p["MEL_Recall"],
        })

df_robustness = pd.DataFrame(robustness_records)
df_robustness.to_csv(ARTIFACT_DIR / "robustness_perturbations.csv", index=False)
print(df_robustness.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=df_robustness, x="Perturbation", y="BA_Multi_Recall_Macro", hue="Architecture", ax=ax)
ax.set_title("Model Robustness Under Dermatoscopic Image Perturbations (Macro Recall / Balanced Accuracy)")
ax.set_ylabel("Macro Recall (Balanced Accuracy)")
ax.set_ylim(0.0, 1.0)
plt.xticks(rotation=25, ha="right")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "robustness_perturbations.pdf")
plt.savefig(ARTIFACT_DIR / "robustness_perturbations.png")
plt.show()

## 9. Clinical Error Analysis, Confusion Matrices, Melanoma False Negative Audit & Grad-CAM (Priority 4)
To address **Priority 4**:
1. **Side-by-Side Absolute and Row-Normalized Confusion Matrices** for the flagship model and baselines.
2. **Multi-Class ROC and Precision-Recall (PR) Curves** (visualizing Table 2).
3. **In-Depth Audit of Melanoma False Negatives ($FN_{\text{MEL}}$):** Breaking down which benign/malignant classes absorb missed melanomas, comparing their ABCD feature profiles against True Positives ($TP_{\text{MEL}}$), and plotting representative visual examples of **TP, FP, and FN**.
4. **Grad-CAM Explainability:** Computing spatial activation heatmaps from the final convolutional layer of `EfficientNetV2B0` for representative dermatoscopic lesions.

In [ ]:
# --- 9.1 Absolute & Row-Normalized Confusion Matrices (Priority 4.2) ---
cm_abs = eff_res["cm_opt"]
cm_norm = cm_abs.astype(np.float32) / np.clip(cm_abs.sum(axis=1, keepdims=True), 1, None)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.8))
sns.heatmap(cm_abs, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_CODES, yticklabels=CLASS_CODES, ax=axes[0])
axes[0].set_title("Absolute Confusion Matrix (Counts, N = " + str(len(y_test)) + ")")
axes[0].set_xlabel("Predicted Diagnosis")
axes[0].set_ylabel("Ground-Truth Diagnosis")

sns.heatmap(cm_norm * 100.0, annot=True, fmt=".1f", cmap="Blues", xticklabels=CLASS_CODES, yticklabels=CLASS_CODES, ax=axes[1])
axes[1].set_title("Row-Normalized Confusion Matrix (% Recall per Class)")
axes[1].set_xlabel("Predicted Diagnosis")
axes[1].set_ylabel("Ground-Truth Diagnosis")

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "confusion_matrices.pdf")
plt.savefig(ARTIFACT_DIR / "confusion_matrices.png")
plt.show()

# --- 9.2 ROC-AUC and PR-AUC Curves Across All 7 Classes ---
y_test_bin = label_binarize(y_test, classes=np.arange(N_CLASSES))
probs_opt = eff_res["test_probs"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.8))
colors = plt.cm.tab10(np.linspace(0, 1, N_CLASSES))

for k, color in zip(range(N_CLASSES), colors):
    fpr, tpr, _ = roc_curve(y_test_bin[:, k], probs_opt[:, k])
    r_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, color=color, lw=2, label=f"{CLASS_CODES[k]} (AUC = {r_auc:.3f})")
    
    prec_c, rec_c, _ = precision_recall_curve(y_test_bin[:, k], probs_opt[:, k])
    p_auc = average_precision_score(y_test_bin[:, k], probs_opt[:, k])
    axes[1].plot(rec_c, prec_c, color=color, lw=2, label=f"{CLASS_CODES[k]} (PR-AUC = {p_auc:.3f})")

axes[0].plot([0, 1], [0, 1], "k--", lw=1.5)
axes[0].set_title("One-vs-Rest ROC Curves (Test Set)")
axes[0].set_xlabel("False Positive Rate (1 - Specificity)")
axes[0].set_ylabel("True Positive Rate (Sensitivity)")
axes[0].legend(loc="lower right", fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].set_title("One-vs-Rest Precision-Recall (PR) Curves (Test Set)")
axes[1].set_xlabel("Recall (Sensitivity)")
axes[1].set_ylabel("Precision (PPV)")
axes[1].legend(loc="lower left", fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "roc_pr_curves.pdf")
plt.savefig(ARTIFACT_DIR / "roc_pr_curves.png")
plt.show()

# --- 9.3 Melanoma False Negative (FN) Clinical Audit (Priority 4.1 & 4.4) ---
preds_opt = eff_res["test_preds_opt"]
mel_true_mask = (y_test == MEL_IDX)
mel_tp_idx = np.where(mel_true_mask & (preds_opt == MEL_IDX))[0]
mel_fn_idx = np.where(mel_true_mask & (preds_opt != MEL_IDX))[0]
mel_fp_idx = np.where((~mel_true_mask) & (preds_opt == MEL_IDX))[0]

print(f"\n--- Clinical Audit of Melanoma (MEL, Support = {int(np.sum(mel_true_mask))}) ---")
print(f"  True Positives  (TP_MEL): {len(mel_tp_idx)} ({len(mel_tp_idx)/max(1, np.sum(mel_true_mask))*100:.1f}%)")
print(f"  False Negatives (FN_MEL): {len(mel_fn_idx)} ({len(mel_fn_idx)/max(1, np.sum(mel_true_mask))*100:.1f}%)")
print(f"  False Positives (FP_MEL): {len(mel_fp_idx)}")

fn_confused_with = Counter([CLASS_CODES[p] for p in preds_opt[mel_fn_idx]])
print("  Classes absorbing Missed Melanomas (FN_MEL):", dict(fn_confused_with))
fp_origin_classes = Counter([CLASS_CODES[t] for t in y_test[mel_fp_idx]])
print("  True classes of Melanoma False Alarms (FP_MEL):", dict(fp_origin_classes))

# Compare ABCD descriptor profiles between TP_MEL and FN_MEL
X_feat_test = X_feat_dr[test_idx]
df_mel_audit = pd.DataFrame({
    "Feature": FEATURE_NAMES,
    "TP_MEL_Mean": X_feat_test[mel_tp_idx].mean(axis=0) if len(mel_tp_idx) > 0 else np.nan,
    "FN_MEL_Mean": X_feat_test[mel_fn_idx].mean(axis=0) if len(mel_fn_idx) > 0 else np.nan,
    "NV_True_Mean": X_feat_test[y_test == NV_IDX].mean(axis=0),
})
df_mel_audit.to_csv(ARTIFACT_DIR / "melanoma_fn_feature_audit.csv", index=False)

# --- 9.4 Grad-CAM Visual Explanations (Priority 4.3) ---
def compute_gradcam_heatmap(full_model, backbone_model, img_u8, tab_feat, target_class_idx):
    """
    Computes Grad-CAM heatmap with respect to the last convolutional activation map of the backbone.
    """
    # Find last 4D convolutional output layer inside backbone
    last_conv_layer = None
    for layer in reversed(backbone_model.layers):
        try:
            if len(layer.output.shape) == 4:
                last_conv_layer = layer
                break
        except Exception:
            continue
    if last_conv_layer is None:
        return np.zeros((IMG_SIZE[0], IMG_SIZE[1]), dtype=np.float32)

    sub_backbone = tf.keras.Model(inputs=backbone_model.inputs, outputs=[last_conv_layer.output, backbone_model.output])
    
    img_tensor = tf.cast(np.expand_dims(img_u8, axis=0), tf.float32)
    tab_tensor = tf.cast(np.expand_dims(tab_feat, axis=0), tf.float32)
    
    with tf.GradientTape() as tape:
        conv_out, bb_out = sub_backbone(img_tensor, training=False)
        tape.watch(conv_out)
        # Re-run classification head layers on top of conv_out/bb_out
        x1 = full_model.get_layer("gap")(conv_out)
        x1 = full_model.get_layer("vis_dropout")(x1, training=False)
        x1 = full_model.get_layer("vis_dense")(x1)
        x2 = full_model.get_layer("tab_dense")(tab_tensor)
        comb = full_model.get_layer("hybrid_concat")([x1, x2])
        comb = full_model.get_layer("head_dropout")(comb, training=False)
        preds = full_model.get_layer("Classifier_Softmax")(comb)
        class_Score = preds[:, target_class_idx]

    grads = tape.gradient(class_Score, conv_out)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_out_val = conv_out[0]
    heatmap = tf.reduce_sum(tf.multiply(pooled_grads, conv_out_val), axis=-1)
    heatmap = np.maximum(heatmap.numpy(), 0)
    if np.max(heatmap) > 0:
        heatmap /= np.max(heatmap)
    heatmap_resized = cv2.resize(heatmap, (IMG_SIZE[1], IMG_SIZE[0]))
    return heatmap_resized

# Select representative TP, FN, and FP cases for Melanoma Grad-CAM visualization
rep_cases = []
if len(mel_tp_idx) > 0:
    rep_cases.append(("True Positive (TP: MEL -> MEL)", mel_tp_idx[0]))
if len(mel_fn_idx) > 0:
    rep_cases.append((f"False Negative (FN: MEL -> {CLASS_CODES[preds_opt[mel_fn_idx[0]]]})", mel_fn_idx[0]))
if len(mel_fp_idx) > 0:
    rep_cases.append((f"False Positive (FP: {CLASS_CODES[y_test[mel_fp_idx[0]]]} -> MEL)", mel_fp_idx[0]))

if rep_cases:
    fig, axes = plt.subplots(len(rep_cases), 3, figsize=(11, 3.6 * len(rep_cases)))
    if len(rep_cases) == 1:
        axes = np.expand_dims(axes, axis=0)
    for row_i, (title_str, local_te_idx) in enumerate(rep_cases):
        img_u8 = X_test_imgs[local_te_idx]
        tab_f = f_test_scaled[local_te_idx]
        pred_cls = int(preds_opt[local_te_idx])
        conf_val = float(probs_opt[local_te_idx, pred_cls])
        
        hm = compute_gradcam_heatmap(eff_res["model"], eff_res["backbone"], img_u8, tab_f, pred_cls)
        hm_uint8 = np.uint8(255 * hm)
        jet = cv2.cvtColor(cv2.applyColorMap(hm_uint8, cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
        overlay = np.clip(0.6 * img_u8 + 0.4 * jet, 0, 255).astype(np.uint8)
        
        axes[row_i, 0].imshow(img_u8)
        axes[row_i, 0].set_title(f"{title_str}\nDermoscopic Input")
        axes[row_i, 0].axis("off")
        
        axes[row_i, 1].imshow(hm, cmap="jet")
        axes[row_i, 1].set_title(f"Grad-CAM Activation Map\n(Target={CLASS_CODES[pred_cls]}, p={conf_val:.2f})")
        axes[row_i, 1].axis("off")
        
        axes[row_i, 2].imshow(overlay)
        axes[row_i, 2].set_title("Clinical Region Overlay")
        axes[row_i, 2].axis("off")
        
    plt.tight_layout()
    plt.savefig(ARTIFACT_DIR / "gradcam_analysis.pdf")
    plt.savefig(ARTIFACT_DIR / "gradcam_analysis.png")
    plt.show()

# Save per-class detailed tables (Table 2 & Table 3)
eff_res["df_cls_opt"].to_csv(ARTIFACT_DIR / "table3_per_class_metrics.csv", index=False)
print("\n--- Table 3: Per-Class Diagnostic Performance (DL Optimized) ---")
print(eff_res["df_cls_opt"].to_string(index=False))
# --- 9.5 Automatic LaTeX Table Snippet Export for BSPC Manuscript ---
latex_out = [
    "% Auto-generated LaTeX tables from Kaggle execution",
    df_progression.to_latex(index=False, float_format="%.3f"),
    df_seed_agg.to_latex(index=False, float_format="%.3f"),
    eff_res["df_cls_opt"].to_latex(index=False, float_format="%.3f"),
    df_costs.to_latex(index=False, float_format="%.2f"),
    df_robustness.to_latex(index=False, float_format="%.3f"),
]
with open(ARTIFACT_DIR / "auto_generated_tables.tex", "w") as f:
    for block in latex_out:
        f.write(block + chr(10) + chr(10))

print(f"All tables (.csv & .tex), figures (.pdf & .png), weights, and reproducibility logs exported to: {ARTIFACT_DIR}")